# Construyendo un Agente Básico

En este notebook crearemos un agente simple desde cero (sin frameworks) para entender los conceptos fundamentales. Luego veremos cómo usar un framework real.

```{note}
Este notebook es **educativo**. Para producción, usa frameworks como LangChain o CrewAI.
```

## Parte 1: Agente conceptual (sin API)

Primero, simulemos cómo piensa un agente usando solo Python puro. Esto ayuda a entender la lógica sin depender de una API.

In [ ]:
# --- Definimos herramientas (funciones Python) ---

def calcular_volumen_concreto(largo_m: float, ancho_m: float, alto_m: float) -> float:
    """Calcula el volumen de concreto en m³."""
    vol = largo_m * ancho_m * alto_m
    return round(vol, 3)

def calcular_acero_columna(seccion_cm: float, altura_m: float, cuantia: float = 0.01) -> dict:
    """Calcula el acero de refuerzo de una columna.
    cuantia: fracción del área de la sección (mínimo 1% según NSR-10).
    """
    area_seccion_cm2 = seccion_cm ** 2
    area_acero_cm2 = area_seccion_cm2 * cuantia
    # Barras #5 (diámetro 1.59cm, área 1.99 cm²)
    n_barras = int(area_acero_cm2 / 1.99) + 1
    peso_kg = n_barras * (1.59 / 100) ** 2 * 3.1416 / 4 * 7850 * altura_m
    return {
        "area_acero_cm2": round(area_acero_cm2, 2),
        "n_barras_no5": n_barras,
        "peso_kg": round(peso_kg, 2)
    }

def precio_unitario(material: str) -> dict:
    """Consulta precios unitarios (simulados, COP 2024)."""
    precios = {
        "concreto_4000psi": {"unidad": "m³", "precio_cop": 650_000},
        "acero_60ksi":      {"unidad": "kg", "precio_cop": 5_200},
        "formaleta":        {"unidad": "m²", "precio_cop": 35_000},
        "mano_obra":        {"unidad": "m³", "precio_cop": 180_000},
    }
    return precios.get(material, {"error": f"Material '{material}' no encontrado"})

print("Herramientas definidas ✓")
print(f"  - calcular_volumen_concreto(0.3, 0.3, 3.0) = {calcular_volumen_concreto(0.3, 0.3, 3.0)} m³")
print(f"  - calcular_acero_columna(30, 3.0) = {calcular_acero_columna(30, 3.0)}")
print(f"  - precio_unitario('concreto_4000psi') = {precio_unitario('concreto_4000psi')}")

In [ ]:
# --- Simulamos el razonamiento del agente ---

def agente_apu_columna(seccion_cm: float, altura_m: float):
    """Agente que genera un APU básico de una columna.
    Simula el proceso: planificar → usar herramientas → generar resultado.
    """
    print("="*60)
    print("AGENTE: Generador de APU para Columnas")
    print("="*60)
    
    # PASO 1: Planificación
    print("\n[Paso 1] PLANIFICACIÓN")
    print(f"  Tarea: APU columna {seccion_cm}x{seccion_cm}cm, altura {altura_m}m")
    print("  Plan:")
    print("    1. Calcular volumen de concreto")
    print("    2. Calcular acero de refuerzo")
    print("    3. Calcular formaleta")
    print("    4. Consultar precios unitarios")
    print("    5. Armar APU")
    
    # PASO 2: Ejecución con herramientas
    print("\n[Paso 2] EJECUCIÓN")
    
    lado_m = seccion_cm / 100
    
    # Herramienta 1: Volumen
    vol = calcular_volumen_concreto(lado_m, lado_m, altura_m)
    print(f"  → Volumen concreto: {vol} m³")
    
    # Herramienta 2: Acero
    acero = calcular_acero_columna(seccion_cm, altura_m)
    print(f"  → Acero: {acero['n_barras_no5']} barras #5, {acero['peso_kg']} kg")
    
    # Herramienta 3: Formaleta
    formaleta_m2 = round(4 * lado_m * altura_m, 2)
    print(f"  → Formaleta: {formaleta_m2} m²")
    
    # Herramienta 4: Precios
    p_concreto = precio_unitario("concreto_4000psi")["precio_cop"]
    p_acero = precio_unitario("acero_60ksi")["precio_cop"]
    p_formaleta = precio_unitario("formaleta")["precio_cop"]
    p_mo = precio_unitario("mano_obra")["precio_cop"]
    
    # PASO 3: Generar APU
    print("\n[Paso 3] RESULTADO — APU Columna")
    print("-"*60)
    print(f"{'Ítem':<25} {'Cantidad':>10} {'Unidad':>6} {'P.Unit':>12} {'Subtotal':>12}")
    print("-"*60)
    
    items = [
        ("Concreto 4000 PSI", vol, "m³", p_concreto),
        ("Acero 60 ksi", acero["peso_kg"], "kg", p_acero),
        ("Formaleta", formaleta_m2, "m²", p_formaleta),
        ("Mano de obra", vol, "m³", p_mo),
    ]
    
    total = 0
    for nombre, cant, unidad, precio in items:
        subtotal = cant * precio
        total += subtotal
        print(f"{nombre:<25} {cant:>10.2f} {unidad:>6} {precio:>12,.0f} {subtotal:>12,.0f}")
    
    print("-"*60)
    print(f"{'TOTAL':.<51} {total:>12,.0f} COP")
    print(f"{'COSTO POR m³':.<51} {total/vol:>12,.0f} COP/m³")
    
    # PASO 4: Reflexión
    print("\n[Paso 4] REFLEXIÓN")
    if acero["area_acero_cm2"] / (seccion_cm**2) >= 0.01:
        print("  ✓ Cuantía de acero cumple mínimo NSR-10 (1%)")
    else:
        print("  ✗ ALERTA: Cuantía de acero NO cumple mínimo NSR-10")
    
    return {"total_cop": total, "costo_m3": total/vol}

# Ejecutar el agente
resultado = agente_apu_columna(seccion_cm=30, altura_m=3.0)

## Parte 2: Patrón ReAct

El patrón **ReAct** (Reasoning + Acting) es cómo los agentes modernos combinan razonamiento con acciones:

```
Pensamiento: Necesito calcular el volumen de concreto de la columna.
Acción: calcular_volumen_concreto(0.3, 0.3, 3.0)
Observación: 0.27 m³

Pensamiento: Ahora necesito el acero. Usaré cuantía mínima del 1%.
Acción: calcular_acero_columna(30, 3.0, 0.01)
Observación: {n_barras: 5, peso_kg: 4.68}

Pensamiento: Tengo todos los datos. Puedo generar el APU.
Acción: generar_reporte()
```

Cada ciclo **Pensamiento → Acción → Observación** es un paso del agente. El LLM decide cuál herramienta usar basándose en el razonamiento.

## Parte 3: Conectando con un LLM real (opcional)

Si tienes acceso a una API de LLM, así es como se vería la conexión con herramientas. Este código requiere una API key del proveedor que elijas.

```python
from openai import OpenAI  # o cualquier SDK de LLM

client = OpenAI()  # usa la variable de entorno del proveedor

# Definir herramientas para el LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "calcular_volumen_concreto",
            "description": "Calcula volumen de concreto en m³",
            "parameters": {
                "type": "object",
                "properties": {
                    "largo_m": {"type": "number"},
                    "ancho_m": {"type": "number"},
                    "alto_m": {"type": "number"}
                },
                "required": ["largo_m", "ancho_m", "alto_m"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "precio_unitario",
            "description": "Consulta precio unitario de materiales de construcción",
            "parameters": {
                "type": "object",
                "properties": {
                    "material": {"type": "string"}
                },
                "required": ["material"]
            }
        }
    }
]

# El LLM decide qué herramientas usar
response = client.chat.completions.create(
    model="gpt-4o-mini",  # o el modelo que prefieras
    tools=tools,
    messages=[{
        "role": "user",
        "content": "Calcula el costo del concreto para una columna de 30x30cm y 3m de alto"
    }]
)
```

El LLM responderá pidiendo usar la herramienta `calcular_volumen_concreto` con los parámetros correctos. La estructura de herramientas es similar en todos los proveedores (OpenAI, Google, Anthropic, etc.).

## Ejercicio propuesto

1. Modifica `agente_apu_columna` para que acepte un parámetro `tipo_concreto` que puede ser `"3000psi"`, `"4000psi"` o `"5000psi"` con precios diferentes.
2. Agrega una herramienta `calcular_estribos` que calcule la cantidad de estribos dado un espaciamiento.
3. Incluye el costo de los estribos en el APU final.

*Pista: los estribos #3 a cada 15cm en una columna de 3m serían 3.0/0.15 = 20 estribos.*